In [ ]:
# !pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.6 MB/s eta 0:00:00


In [ ]:
!mkdir -p ~/.kaggle/
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d gowrishankarp/newspaper-text-summarization-cnn-dailymail

Dataset URL: https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail
License(s): CC0-1.0
 94% 474M/503M [00:01<00:00, 358MB/s]
100% 503M/503M [00:01<00:00, 430MB/s]


In [ ]:
from datasets import load_dataset

In [ ]:
!unzip /content/newspaper-text-summarization-cnn-dailymail.zip

Archive:  /content/newspaper-text-summarization-cnn-dailymail.zip
replace cnn_dailymail/test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace cnn_dailymail/train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace cnn_dailymail/validation.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
data_files = {
    "train": "/content/cnn_dailymail/train.csv",
    "test": "/content/cnn_dailymail/test.csv",
    "validation": "/content/cnn_dailymail/validation.csv"
}
dataset = load_dataset("csv", data_files=data_files)
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'article', 'highlights'],
        num_rows: 287113
    })
    test: Dataset({
        features: ['id', 'article', 'highlights'],
        num_rows: 11490
    })
    validation: Dataset({
        features: ['id', 'article', 'highlights'],
        num_rows: 13368
    })
})


In [ ]:
def format_data(data):
  instruction = "Summarize the following news article:\n\n"
  return {
      "text": f"{instruction}{data['article']}\n\n###Summary:{data['highlights']}"
  }

In [ ]:
dataset = dataset.map(format_data)

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'article', 'highlights', 'text'],
        num_rows: 287113
    })
    test: Dataset({
        features: ['id', 'article', 'highlights', 'text'],
        num_rows: 11490
    })
    validation: Dataset({
        features: ['id', 'article', 'highlights', 'text'],
        num_rows: 13368
    })
})

In [ ]:
# import transformers as tf
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
import peft
import torch

In [ ]:
model_name="Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    encoded = tokenizer(
        examples["text"],            # <---- FIXED, no [0]
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # labels = input_ids, with padding masked
    labels = []
    for ids in encoded["input_ids"]:
        labels.append([
            id if id != tokenizer.pad_token_id else -100
            for id in ids
        ])

    encoded["labels"] = labels
    return encoded

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(['id', 'article', 'highlights', 'text'])

In [ ]:
tokenized_dataset.save_to_disk("/content/tokenized_dataset")

Saving the dataset (0/4 shards):   0%|          | 0/287113 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/11490 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13368 [00:00<?, ? examples/s]

In [ ]:
!zip -r /content/tokenized_dataset.zip /content/tokenized_dataset

  adding: content/tokenized_dataset/ (stored 0%)
  adding: content/tokenized_dataset/train/ (stored 0%)
  adding: content/tokenized_dataset/train/data-00000-of-00004.arrow (deflated 74%)
  adding: content/tokenized_dataset/train/state.json (deflated 60%)
  adding: content/tokenized_dataset/train/data-00001-of-00004.arrow (deflated 74%)
  adding: content/tokenized_dataset/train/data-00002-of-00004.arrow (deflated 74%)
  adding: content/tokenized_dataset/train/data-00003-of-00004.arrow (deflated 74%)
  adding: content/tokenized_dataset/train/dataset_info.json (deflated 70%)
  adding: content/tokenized_dataset/test/ (stored 0%)
  adding: content/tokenized_dataset/test/state.json (deflated 37%)
  adding: content/tokenized_dataset/test/dataset_info.json (deflated 70%)
  adding: content/tokenized_dataset/test/data-00000-of-00001.arrow (deflated 74%)
  adding: content/tokenized_dataset/dataset_dict.json (deflated 2%)
  adding: content/tokenized_dataset/validation/ (stored 0%)
  adding: conten

In [ ]:
sample_train_tokenized = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
sample_validation_tokenized = tokenized_dataset["validation"].shuffle(seed=42).select(range(1000))

In [ ]:
len(sample_train_tokenized['labels'][0])

512

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    )

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 32,
    target_modules = ["q_proj", "v_proj","k_proj","o_proj"],
    lora_dropout = 0.05,
    task_type = TaskType.CAUSAL_LM,
    bias = "none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
training_arg = TrainingArguments(
    output_dir = "/content/drive/MyDrive/qwen3_lora_final",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    optim = "paged_adamw_32bit",
    save_steps = 100,
    logging_steps = 10,
    learning_rate = 1e-5,
    fp16 = True,
    bf16 = False,
    max_grad_norm = 0.3,
    num_train_epochs =3,
    eval_strategy="steps",
    eval_steps=100,
    gradient_checkpointing=True,
    report_to="none",
    save_strategy="steps",
    save_total_limit=3,
    remove_unused_columns=False,
)

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_arg,
    train_dataset = sample_train_tokenized,
    eval_dataset = sample_validation_tokenized,
    data_collator = data_collator,
)

In [ ]:
model.config.use_cache = False  # Disable cache to use gradient checkpointing

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
100,2.452720,2.454866
200,2.403489,2.420836


Step,Training Loss,Validation Loss
100,2.452720,2.454866
200,2.403489,2.420836


KeyboardInterrupt: 

In [ ]:
trainer.train(resume_from_checkpoint="/content/drive/MyDrive/qwen3_lora_final/checkpoint-200")

Step,Training Loss,Validation Loss
300,2.395988,2.393211
400,2.381047,2.384712
500,2.355412,2.381055
600,2.342939,2.378814
700,2.334607,2.377348
800,2.337751,2.376338
900,2.359890,2.375995


TrainOutput(global_step=939, training_loss=1.8590576549688467, metrics={'train_runtime': 7006.4949, 'train_samples_per_second': 2.141, 'train_steps_per_second': 0.134, 'total_flos': 6.079540757004288e+16, 'train_loss': 1.8590576549688467, 'epoch': 3.0})

In [ ]:
save_path = "/content/drive/MyDrive/qwen3_lora_progress"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/qwen3_lora_progress/tokenizer_config.json',
 '/content/drive/MyDrive/qwen3_lora_progress/chat_template.jinja',
 '/content/drive/MyDrive/qwen3_lora_progress/tokenizer.json')